# 01 – Alle Citi Bike-data automatisch downloaden en samenvatten

| | |
|---|---|
| **Auteur(s)** | Kobe Vandenberk |
| **Wat** | **Alle** ritten sinds de start van Citi Bike (juni 2013) automatisch van de officiële bron halen, controleren en in één passage samenvatten tot *vertrekken per station per uur*. Daarnaast het uurlijkse weer in NYC over dezelfde periode. |
| **Input** | alle NYC-bestanden in `https://s3.amazonaws.com/tripdata/`, Open-Meteo archive API |
| **Output** | `data/raw/*.zip` + `manifest.json`, `data/station_hour/<periode>.parquet`, `data/stations.parquet`, `data/audit.json`, `data/trips_sample.parquet`, `data/weather.parquet` |
| **GenAI** | Code en tekst opgesteld met hulp van Claude; keuzes en resultaten nagekeken door Kobe. |

## Onderzoeksvraag van deze reeks notebooks
**Hoeveel fietsen vertrekken er in een bepaalde zone van New York in een bepaald uur?**

Dat is de vraag die Citi Bike elke dag moet beantwoorden om fietsen te *herverdelen*: lege stations in de spits betekenen dat mensen de auto, taxi of metro nemen. Een goede voorspelling van de vraag helpt dus rechtstreeks het thema **Going green**.

## Twee studies
1. **Modelselectie op 2024** (`02`, `03`, `05b`–`05d`, mappen `data/2024` en `models/2024`): op één jaar vergeleken we baseline, GLM, random forest, PyCaret en gradient boosting. **Gradient boosting met Poisson-verlies** won.
2. **Alle data** (dit notebook, `02b`, `04`, `05a`, `05e`, `06`): met de gekozen techniek trainen we op álle ritten sinds 2013. Meer jaren betekent meer voorbeelden van zeldzame situaties (feestdagen, stormen, sneeuw) — precies waar het 2024-model de grootste fouten maakte.

## Wat staat er in de bucket?
De bestanden zijn niet allemaal hetzelfde opgebouwd:
- **2013–2023**: één zip per jaar, met daarin mappen per maand (en soms dezelfde maand nog eens als los bestand);
- **vanaf 2024**: één zip per maand, met meerdere CSV-delen van ± 1 miljoen rijen;
- **`JC-…`**: Jersey City, een apart systeem. Die laten we weg (ook in de cleaning tellen JC-stations niet mee).

Ook de kolommen veranderden door de jaren (`starttime` → `Start Time` → `started_at`, `usertype` → `member_casual`, oude station-id's zonder punt, geen ritnummer vóór 2021). `citibike.py` zet alles om naar het schema van vandaag; `02b_eda_all_years` toont die verschillen.

## 0. Imports

In [1]:
import json

import pandas as pd

import citibike as cb

pd.set_option("display.max_columns", 30)
print("Data-map:", cb.DATA_DIR)

Data-map: C:\Users\kobev\Documents\semester 1\3APPAI - Deep Learning\CloudAI-Challenge\citi_bike\Kobe\data


## 1. De bucket uitlezen
We vragen de lijst van alle bestanden op (de S3-API geeft er telkens 1.000, dus we volgen de *continuation token*) en houden de NYC-bestanden over.

In [2]:
bucket = cb.list_bucket()
files = cb.nyc_files(bucket)
print(f"Bestanden in de bucket: {len(bucket)}, waarvan {len(files)} NYC-bestanden "
      f"({files['bytes'].sum() / 1e9:.1f} GB) en {bucket['file'].str.startswith('JC').sum()} Jersey City-bestanden.")
files.assign(GB=(files["bytes"] / 1e9).round(2))[["file", "GB", "last_modified"]]

Bestanden in de bucket: 178, waarvan 44 NYC-bestanden (32.5 GB) en 133 Jersey City-bestanden.


,file,GB,last_modified
0,2013-citibike-tripdata.zip,0.33,2024-02-22T14:26:20.000Z
1,2014-citibike-tripdata.zip,0.22,2024-02-22T14:26:20.000Z
2,2015-citibike-tripdata.zip,0.28,2024-02-22T14:26:20.000Z
3,2016-citibike-tripdata.zip,0.44,2024-02-22T14:26:20.000Z
4,2017-citibike-tripdata.zip,0.56,2024-02-22T14:26:20.000Z
5,2018-citibike-tripdata.zip,1.34,2024-02-22T14:26:20.000Z
6,2019-citibike-tripdata.zip,0.86,2024-02-22T14:52:46.000Z
7,2020-citibike-tripdata.zip,0.75,2024-07-31T18:18:37.000Z
8,2021-citibike-tripdata.zip,1.07,2024-08-13T12:47:01.000Z
9,2022-citibike-tripdata.zip,1.23,2024-07-31T18:18:37.000Z


## 2. Downloaden (met controle)
Per bestand: downloaden naar een `.part`-bestand en pas hernoemen als de grootte klopt (een afgebroken download wordt zo nooit voor een geldige zip aangezien), daarna een **SHA-256 checksum** in `manifest.json`. Bestanden die al volledig aanwezig zijn, worden niet opnieuw gedownload. Het manifest vermeldt ook welke CSV's er in elke zip zitten (ook in geneste zip's).

In [3]:
manifest = cb.download_all()
manifest.assign(GB=(manifest["bytes"] / 1e9).round(2), csvs=manifest["csv_files"].str.len())[["file", "GB", "csvs", "status"]]

2013-citibike-tripdata.zip           al aanwezig  0.33 GB,  17 CSV's (0 s)


2014-citibike-tripdata.zip           al aanwezig  0.22 GB,  12 CSV's (0 s)

2015-citibike-tripdata.zip           al aanwezig  0.28 GB,  16 CSV's (0 s)


2016-citibike-tripdata.zip           al aanwezig  0.44 GB,  20 CSV's (0 s)


2017-citibike-tripdata.zip           al aanwezig  0.56 GB,  20 CSV's (0 s)


...

202606-citibike-tripdata.zip         al aanwezig  1.05 GB,   6 CSV's (0 s)


202607-citibike-tripdata.zip         al aanwezig  0.97 GB,   5 CSV's (0 s)


202608-citibike-tripdata.zip         al aanwezig  1.02 GB,   6 CSV's (0 s)


202609-citibike-tripdata.zip         al aanwezig  0.97 GB,   5 CSV's (0 s)


,file,GB,csvs,status
0,2013-citibike-tripdata.zip,0.33,17,al aanwezig
1,2014-citibike-tripdata.zip,0.22,12,al aanwezig
2,2015-citibike-tripdata.zip,0.28,16,al aanwezig
3,2016-citibike-tripdata.zip,0.44,20,al aanwezig
4,2017-citibike-tripdata.zip,0.56,20,al aanwezig
5,2018-citibike-tripdata.zip,1.34,35,al aanwezig
6,2019-citibike-tripdata.zip,0.86,26,al aanwezig
7,2020-citibike-tripdata.zip,0.75,27,al aanwezig
8,2021-citibike-tripdata.zip,1.07,36,al aanwezig
9,2022-citibike-tripdata.zip,1.23,36,al aanwezig


## 3. Eén passage door alle bestanden
`aggregate_all` leest elk bestand in stukken van 500.000 rijen en houdt per stuk enkel bij:
- het aantal vertrekken per station per uur (+ hoeveel daarvan members / e-bikes waren);
- een stationslijst (naam, coördinaten, eerste en laatste gebruik);
- een **audit per maand**: rijen, ontbrekende waarden, duplicaten, wat we weglieten;
- een **ruwe steekproef van 0,2%** van alle ritten voor de EDA.

Per bronbestand worden de resultaten apart weggeschreven (`data/station_hour/2013.parquet`, …): zo blijft het geheugengebruik beperkt en kan een onderbroken run hervatten. Per maand gebruiken we maar **één versie** van de data (`select_csvs`): de 2013-zip bevat elke maand twee keer.

In [4]:
result = cb.aggregate_all()

2013-citibike-tripdata.zip           al verwerkt


2014-citibike-tripdata.zip           al verwerkt


2015-citibike-tripdata.zip           al verwerkt


2016-citibike-tripdata.zip           al verwerkt


2017-citibike-tripdata.zip           al verwerkt


...

202605-citibike-tripdata.zip         al verwerkt


202606-citibike-tripdata.zip         al verwerkt


202607-citibike-tripdata.zip         al verwerkt


202608-citibike-tripdata.zip         al verwerkt


202609-citibike-tripdata.zip         al verwerkt


## 4. Wat hebben we nu?

In [5]:
audit = pd.DataFrame([a for a in result["audit"] if "rows" in a])
per_year = (audit.assign(jaar=audit["month"].str[:4]).groupby("jaar")
                 [["rows", "departures", "no_start_station", "outside_nyc", "duplicate_rows", "duration_negative"]].sum())
per_year

,rows,departures,no_start_station,outside_nyc,duplicate_rows,duration_negative
jaar,,,,,,
2013,5614888,5614888,0,0,177,0
2014,8081216,8081216,0,0,211,0
2015,9937969,9937969,0,0,42118,70
2016,13845655,13845652,0,3,7349,83
2017,16364657,16364532,0,125,6539,49
2018,17548339,17545761,2497,81,0,0
2019,20551697,20551517,180,0,0,0
2020,19562314,19562197,111,6,0,0
2021,27130122,27129629,493,0,0,278


In [6]:
totals = audit[["rows", "departures"]].sum()
dropped = [a for a in result["audit"] if "rows" not in a]
print(f"Ruwe rijen {audit['month'].min()}–{audit['month'].max()}: {totals['rows']:>13,}")
print(f"Geldige vertrekken:                 {totals['departures']:>13,}  ({totals['departures'] / totals['rows']:.3%})")
print(f"Stations (alle id's ooit):          {len(result['stations']):>13,}")
print(f"Weggelaten dubbele CSV-versies:     {sum(len(a['csv_files']) for a in dropped):>13}")

Ruwe rijen 201306–202609:   328,800,493
Geldige vertrekken:                   328,708,160  (99.972%)
Stations (alle id's ooit):                  3,824
Weggelaten dubbele CSV-versies:                21


**Interpretatie:**
- **328,8 miljoen ritten** tussen juni 2013 en september 2026, waarvan **99,97%** bruikbaar is als vertrek. Het aantal ritten per jaar steeg van 5,6 miljoen (2013, zeven maanden) naar 45,8 miljoen (2025).
- **21 dubbele CSV-versies** zijn weggelaten: de zip's van 2013 en 2018 bevatten elke maand twee keer (los bestand én in een maandmap). Zonder die controle zouden we die jaren dubbel tellen. Het totaal voor 2018 (17,5 miljoen) klopt met het officiële jaartotaal van Citi Bike.
- **"Dubbele rijen"** komen alleen voor in 2013–2017, vooral in 2015 (42.118 van 9,9 miljoen). Die jaren hebben datums tot op de minuut of seconde en geen ritnummer, dus twee mensen die samen vertrekken en aankomen (een groepsrit) zien er identiek uit. We laten ze daarom staan: het zijn waarschijnlijk echte ritten, en het gaat om 0,4% in het slechtste jaar.
- **Negatieve ritduren** (± 300 per jaar) vallen telkens in november: het terugzetten van de klok, zoals in 2024 (zie `02_eda_cleaning`). Ze tellen gewoon als vertrek.

## 5. Weer ophalen
Zelfde bron en werkwijze als voor 2024 (Open-Meteo, lokale tijd, zomertijd opgevangen), nu over de hele periode. Per jaar opgevraagd zodat elk antwoord klein blijft. Nieuw ten opzichte van 2024: de **sneeuwhoogte** (cm sneeuw op de grond), toegevoegd na de foutenanalyse in `09`.

In [7]:
first = pd.Timestamp(audit["month"].min() + "01").strftime("%Y-%m-%d")
last = (pd.Timestamp(audit["month"].max() + "01") + pd.offsets.MonthEnd(0)).strftime("%Y-%m-%d")
weather = cb.download_weather_range(first, last)
print(first, "->", last, weather.shape)
assert weather.drop(columns="time").notna().all().all()
weather.describe().round(1)

2013-06-01 -> 2026-09-30 (116880, 7)


,time,temperature_c,precipitation_mm,wind_kmh,snowfall_cm,humidity_pct,snow_depth_cm
count,116880,116880.0,116880.0,116880.0,116880.0,116880.0,116880.0
mean,2020-01-30 23:30:00,12.7,0.1,12.5,0.0,70.8,1.3
min,2013-06-01 00:00:00,-22.4,0.0,0.0,0.0,11.0,0.0
25%,2016-09-30 11:45:00,4.6,0.0,7.9,0.0,57.0,0.0
50%,2020-01-30 23:30:00,13.2,0.0,11.4,0.0,73.0,0.0
75%,2023-06-01 11:15:00,21.2,0.0,16.1,0.0,86.0,0.0
max,2026-09-30 23:00:00,41.3,26.2,63.8,4.4,100.0,51.0
std,NaN,10.2,0.6,6.6,0.1,17.9,5.0


## Samenvatting
- Alle data wordt **met code** opgehaald en gecontroleerd (grootte + checksum in `manifest.json`), ook de jaarbestanden met geneste mappen en dubbele versies.
- Door elk bestand in stukken te lezen en per bronbestand weg te schrijven, verwerkt een gewone laptop 329 miljoen ritten (± 45 minuten, hervatbaar).
- Volgende stap: `02b_eda_all_years` (wat is er anders aan de oudere jaren?), daarna `04_prepare_data`.